# 1.3. Сбор данных с различных источников (Лекция_3)


ETL-процесс: Extract, Transform, Load — базовые понятия. Web-scraping и API: инструменты (Requests, BeautifulSoup, Postman) и нюансы юридической стороны. Парсинг файлов: чтение данных из файлов (CSV, Excel, JSON), первичная фильтрация.


## 1. Чтение и обработка CSV-файлов

Цель: Прочитать файл формата CSV, содержащий данные о студентах университета (ФИО, возраст, специальность), и построить отчёт по среднему возрасту студентов разных факультетов.

- Фамилия,Имя,Возраст,Специальность
- Иванов,Александр,21,Программирование
- Петрова,Анна,20,Экономика
- Кузнецов,Сергей,22,Биология
- Сергеев,Дмитрий,21,Программирование
- Романова,Елена,20,Экономика
- Николаев,Павел,23,Биология

Для каждой специальности определить средний возраст студентов и вывести результат в консоль.



In [ ]:
import csv

# Данные для теста
data = """Фамилия,Имя,Возраст,Специальность
Иванов,Александр,21,Программирование
Петрова,Анна,20,Экономика
Кузнецов,Сергей,22,Биология
Сергеев,Дмитрий,21,Программирование
Романова,Елена,20,Экономика
Николаев,Павел,23,Биология"""

# Сохраняем данные в CSV-файл
with open('students.csv', 'w', newline='', encoding='utf-8') as file:
    file.write(data)

# Читаем и обрабатываем CSV-файл
specialty_age_sum = {}  # Словарь для сумм возрастов по специальностям
specialty_count = {}     # Словарь для счёта студентов по специальностям

with open('students.csv', 'r', newline='', encoding='utf-8') as file:
  reader = csv.DictReader(file)
  for row in reader:
    speciality = row['Специальность']
    age = int(row['Возраст'])

    if not speciality in specialty_age_sum:
      specialty_age_sum[speciality] = 0
    if not speciality in specialty_count:
      specialty_count[speciality] = 0

    specialty_age_sum[speciality] += age
    specialty_count[speciality] += 1

# Вывод среднего возраста по специальностям
for spec, total_age in specialty_age_sum.items():
    avg_age = total_age / specialty_count[spec]
    print(f"Средний возраст студентов по специальности '{spec}' составляет {avg_age:.2f} лет")

Средний возраст студентов по специальности 'Программирование' составляет 21.00 лет
Средний возраст студентов по специальности 'Экономика' составляет 20.00 лет
Средний возраст студентов по специальности 'Биология' составляет 22.50 лет


-------------------------------------------------------------

## 2. Сбор данных с веб-страницы (web scraping)

Цель: Спарсить страницу с рейтингом фильмов и собрать первую десятку лучших фильмов вместе с их оценками.

Источник данных о фильмах: https://www.imdb.com/chart/top/

In [ ]:
from bs4 import BeautifulSoup

# Пример HTML-данных
html_doc = """
<div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#1</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0111161/?ref_=chttp_t_1" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Побег из Шоушенка</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1994</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 22m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">16+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.3"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.3</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->3.2M<!-- -->)</span></span><button aria-label="Rate Побег из Шоушенка"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0111161"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Побег из Шоушенка as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#2</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0068646/?ref_=chttp_t_2" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Крестный отец</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1972</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 55m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">16+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.2"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.2</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->2.2M<!-- -->)</span></span><button aria-label="Rate Крестный отец"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0068646"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Крестный отец as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#3</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0468569/?ref_=chttp_t_3" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Тёмный рыцарь</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2008</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 32m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">14+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.1"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.1</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->3.1M<!-- -->)</span></span><button aria-label="Rate Тёмный рыцарь"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0468569"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Тёмный рыцарь as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#4</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0071562/?ref_=chttp_t_4" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Крестный отец 2</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1974</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">3h 22m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">16+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.0"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.0</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->1.5M<!-- -->)</span></span><button aria-label="Rate Крестный отец 2"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0071562"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Крестный отец 2 as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#5</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0050083/?ref_=chttp_t_5" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">12 разгневанных мужчин</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1957</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1h 36m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">16+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.0"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.0</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->976K<!-- -->)</span></span><button aria-label="Rate 12 разгневанных мужчин"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0050083"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark 12 разгневанных мужчин as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#6</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0167260/?ref_=chttp_t_6" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Властелин колец: Возвращение короля</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2003</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">3h 21m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">12+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.0"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.0</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->2.2M<!-- -->)</span></span><button aria-label="Rate Властелин колец: Возвращение короля"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0167260"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Властелин колец: Возвращение короля as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#7</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0108052/?ref_=chttp_t_7" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Список Шиндлера</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1993</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">3h 15m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">16+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 9.0"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">9.0</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->1.6M<!-- -->)</span></span><button aria-label="Rate Список Шиндлера"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0108052"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Список Шиндлера as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#8</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0120737/?ref_=chttp_t_8" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Властелин колец: Братство кольца</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2001</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 58m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">12+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 8.9"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">8.9</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->2.2M<!-- -->)</span></span><button aria-label="Rate Властелин колец: Братство кольца"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0120737"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Властелин колец: Братство кольца as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#9</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0110912/?ref_=chttp_t_9" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Криминальное чтиво</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1994</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 34m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">18+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 8.8"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">8.8</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->2.4M<!-- -->)</span></span><button aria-label="Rate Криминальное чтиво"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0110912"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Криминальное чтиво as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
  <div
    class="ipc-signpost ipc-signpost--accent1 ipc-signpost--left-aligned ListItem_liRankSignpost__0et53 ListItem_liRankFallbackSignpostColor__FcMT1"
    role="presentation" data-testid="title-list-item-ranking">
    <div class="ipc-signpost__text" role="presentation">#10</div>
  </div>
  <div
    class="ipc-title ipc-title--base ipc-title--title ipc-title-link-no-icon ipc-title--on-textPrimary sc-87337ed2-2 dRlLYG cli-title with-margin">
    <a href="/title/tt0060196/?ref_=chttp_t_10" class="ipc-title-link-wrapper" tabindex="0">
      <h3 class="ipc-title__text">Хороший, плохой, злой</h3>
    </a>
  </div>
  <div class="sc-a55f6282-5 bhUIDq cli-title-metadata"><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">1966</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">2h 41m</span><span
      class="sc-a55f6282-6 iMumIM cli-title-metadata-item">12+</span></div><span class="sc-a55f6282-1 ddtLhl">
    <div class="sc-17ce9e4b-0 ddMjUi sc-a55f6282-2 bxNDyb cli-ratings-container" data-testid="ratingGroup--container">
      <span aria-label="IMDb rating: 8.8"
        class="ipc-rating-star ipc-rating-star--base ipc-rating-star--imdb ratingGroup--imdb-rating"
        data-testid="ratingGroup--imdb-rating"><svg width="24" height="24" xmlns="http://www.w3.org/2000/svg"
          class="ipc-icon ipc-icon--star-inline" viewBox="0 0 24 24" fill="currentColor" role="presentation">
          <path
            d="M12 20.1l5.82 3.682c1.066.675 2.37-.322 2.09-1.584l-1.543-6.926 5.146-4.667c.94-.85.435-2.465-.799-2.567l-6.773-.602L13.29.89a1.38 1.38 0 0 0-2.581 0l-2.65 6.53-6.774.602C.052 8.126-.453 9.74.486 10.59l5.147 4.666-1.542 6.926c-.28 1.262 1.023 2.26 2.09 1.585L12 20.099z">
          </path>
        </svg><span class="ipc-rating-star--rating">8.8</span><span class="ipc-rating-star--voteCount">&nbsp;(<!--
          -->886K<!-- -->)</span></span><button aria-label="Rate Хороший, плохой, злой"
        class="ipc-rate-button sc-17ce9e4b-1 flqkCx ratingGroup--user-rating ipc-rate-button--unrated ipc-rate-button--base"
        data-testid="rate-button"><span class="ipc-rating-star ipc-rating-star--base ipc-rating-star--rate"><svg
            xmlns="http://www.w3.org/2000/svg" width="24" height="24" class="ipc-icon ipc-icon--star-border-inline"
            viewBox="0 0 24 24" fill="currentColor" role="presentation">
            <path
              d="M22.724 8.217l-6.786-.587-2.65-6.22c-.477-1.133-2.103-1.133-2.58 0l-2.65 6.234-6.772.573c-1.234.098-1.739 1.636-.8 2.446l5.146 4.446-1.542 6.598c-.28 1.202 1.023 2.153 2.09 1.51l5.818-3.495 5.819 3.509c1.065.643 2.37-.308 2.089-1.51l-1.542-6.612 5.145-4.446c.94-.81.45-2.348-.785-2.446zm-10.726 8.89l-5.272 3.174 1.402-5.983-4.655-4.026 6.141-.531 2.384-5.634 2.398 5.648 6.14.531-4.654 4.026 1.402 5.983-5.286-3.187z">
            </path>
          </svg><span class="ipc-rating-star--rate">Rate</span></span></button>
    </div><button aria-pressed="false" data-testid="inline-watched-button-tt0060196"
      class="ipc-btn ipc-btn--half-padding ipc-btn--left-align-content ipc-btn--default-height ipc-btn--core-base ipc-btn--theme-base ipc-btn--button-radius ipc-btn--on-accent2 ipc-text-button sc-43529d24-0 jmjwiI sc-a55f6282-3 cJURAI"
      tabindex="0" aria-label="Mark Хороший, плохой, злой as watched" aria-disabled="false"><svg
        xmlns="http://www.w3.org/2000/svg" width="24" height="24"
        class="ipc-icon ipc-icon--visibility ipc-btn__icon ipc-btn__icon--pre watched-button--icon ipc-btn__icon--disable-margin"
        viewBox="0 0 24 24" fill="currentColor" role="presentation">
        <path d="M0 0h24v24H0V0z" fill="none"></path>
        <path
          d="M12 6c3.79 0 7.17 2.13 8.82 5.5C19.17 14.87 15.79 17 12 17s-7.17-2.13-8.82-5.5C4.83 8.13 8.21 6 12 6m0-2C7 4 2.73 7.11 1 11.5 2.73 15.89 7 19 12 19s9.27-3.11 11-7.5C21.27 7.11 17 4 12 4zm0 5c1.38 0 2.5 1.12 2.5 2.5S13.38 14 12 14s-2.5-1.12-2.5-2.5S10.62 9 12 9m0-2c-2.48 0-4.5 2.02-4.5 4.5S9.52 16 12 16s4.5-2.02 4.5-4.5S14.48 7 12 7z">
        </path>
      </svg><span class="ipc-btn__text">Mark as watched</span></button>
  </span>
"""

soup = BeautifulSoup(html_doc, 'html.parser')

movie_titles = soup.find_all('h3', class_='ipc-title__text')
movie_ratings = soup.find_all('span', class_='ipc-rating-star--rating')

films_and_ratings = []
min_length = min(len(movie_titles), len(movie_ratings))

for index in range(min_length):
    title = movie_titles[index].text.strip()
    rating = movie_ratings[index].text.strip()
    films_and_ratings.append((title, rating))

for index, (title, rating) in enumerate(films_and_ratings[:10]):
    print(f"{index+1}. Фильм: {title}, Рейтинг: {rating}")

1. Фильм: Побег из Шоушенка, Рейтинг: 9.3
2. Фильм: Крестный отец, Рейтинг: 9.2
3. Фильм: Тёмный рыцарь, Рейтинг: 9.1
4. Фильм: Крестный отец 2, Рейтинг: 9.0
5. Фильм: 12 разгневанных мужчин, Рейтинг: 9.0
6. Фильм: Властелин колец: Возвращение короля, Рейтинг: 9.0
7. Фильм: Список Шиндлера, Рейтинг: 9.0
8. Фильм: Властелин колец: Братство кольца, Рейтинг: 8.9
9. Фильм: Криминальное чтиво, Рейтинг: 8.8
10. Фильм: Хороший, плохой, злой, Рейтинг: 8.8


-------------------------------------------------------------------

## 3. Чтение и работа с файлами формата Excel (.xlsx)

Цель: Открыть файл Excel, содержащий таблицу зарплат сотрудников, сгруппировать зарплаты по должностям и посчитать среднее значение зарплаты для каждой должности.

Задание:

In [ ]:
import pandas as pd
import requests
import io

# Скачиваем Excel-файл напрямую из Google Sheets
file_id = "16dPeHE-qWwP4vCZLynvgCx2n-eW4AzaO"
url = f"https://docs.google.com/spreadsheets/d/{file_id}/export?format=xlsx"

response = requests.get(url)
response.raise_for_status()

# Читаем файл из памяти (без сохранения на диск)
df = pd.read_excel(io.BytesIO(response.content))

print("Загруженные данные:")
print(df)
print()

# Группировка по должностям и расчёт средней зарплаты
result = df.groupby('Должность')['Зарплата'].mean().reset_index()
result.columns = ['Должность', 'Средняя зарплата']
result['Средняя зарплата'] = result['Средняя зарплата'].round(0).astype(int)

print("Средняя зарплата по должностям:")
print(result)

Загруженные данные:
              ФИО    Должность  Зарплата  Стаж_лет
0    Смирнов В.А.     Аналитик     62000         3
1    Козлова Т.Н.     Менеджер     75000         5
2    Новиков К.О.  Разработчик     95000         7
3   Морозова Д.С.     Аналитик     58000         2
4     Волков И.П.     Менеджер     80000         6
5    Зайцева А.В.           HR     48000         4
6    Соколов Р.Е.  Разработчик     90000         8
7   Лебедева О.Ю.           HR     51000         3
8      Попов Г.М.     Аналитик     65000         4
9  Степанова Н.Л.     Менеджер     72000         5

Средняя зарплата по должностям:
     Должность  Средняя зарплата
0           HR             49500
1     Аналитик             61667
2     Менеджер             75667
3  Разработчик             92500


-------------------------------------------------------------

## 4. Трансформация данных JSON в CSV

Цель: Преобразовать данные формата JSON, полученные от API сервиса новостей, в файл CSV для последующего анализа.

Сохранить полученный JSON в виде таблицы CSV с тремя колонками: "Заголовок", "Автор", "Дата публикации".

Задание:

In [ ]:
import json
import csv

# Пример JSON-данных
news_json = [
    {
        "title": "Новости №1",
        "author": "Автор №1",
        "published_at": "2023-01-01T12:00:00Z"
    },
    {
        "title": "Новости №2",
        "author": "Автор №2",
        "published_at": "2023-01-02T14:00:00Z"
    },
    {
        "title": "Новости №3",
        "author": "Автор №3",
        "published_at": "2023-03-02T14:00:00Z"
    },
    {
        "title": "Новости №4",
        "author": "Автор №4",
        "published_at": "2023-04-02T14:00:00Z"
    },
    {
        "title": "Новости №5",
        "author": "Автор №5",
        "published_at": "2023-05-02T14:00:00Z"
    },
    {
        "title": "Новости №6",
        "author": "Автор №1",
        "published_at": "2023-06-02T14:00:00Z"
    }
]

# Конвертируем JSON в CSV (переписано)
json_prop_names = ["Заголовок", "Автор", "Дата публикации"]

with open('news.csv', 'w', newline='', encoding='utf-8') as csvfile:
    writer = csv.writer(csvfile)
    writer.writerow(json_prop_names)

    for single_news in news_json:
        writer.writerow([
            single_news["title"],
            single_news["author"],
            single_news["published_at"]
        ])

# Проверка результата
with open('news.csv', 'r', newline='', encoding='utf-8') as csvfile:
    reader = csv.reader(csvfile)
    for row in reader:
        print(', '.join(row))

Заголовок, Автор, Дата публикации
Новости №1, Автор №1, 2023-01-01T12:00:00Z
Новости №2, Автор №2, 2023-01-02T14:00:00Z
Новости №3, Автор №3, 2023-03-02T14:00:00Z
Новости №4, Автор №4, 2023-04-02T14:00:00Z
Новости №5, Автор №5, 2023-05-02T14:00:00Z
Новости №6, Автор №1, 2023-06-02T14:00:00Z


------------------------------------------------------------

## 5. Работа с файлами формата JSON

Цель: Прочитать файл формата JSON, содержащий информацию о фильмах кинотеатра (название фильма, жанры, продолжительность в минутах), и вывести все фильмы длительностью менее двух часов (менее 120 минут).

Синтетический набор данных:

In [ ]:
import json

# Исходные данные
movies_json = '''
[
    {"title": "Матрица", "genres": ["Action"], "duration_minutes": 136},
    {"title": "Интерстеллар", "genres": ["Sci-Fi"], "duration_minutes": 169},
    {"title": "Форрест Гамп", "genres": ["Drama"], "duration_minutes": 142},
    {"title": "Один дома", "genres": ["Comedy"], "duration_minutes": 103},
    {"title": "Начало", "genres": ["Thriller"], "duration_minutes": 148}
]
'''

# Переводим строку JSON в словарь Python
movies_data = json.loads(movies_json)

min_duration_minutes = 120
short_movies = []
for movie in movies_data:
    if movie["duration_minutes"] < min_duration_minutes:
        short_movies.append(movie["title"])

if len(short_movies) > 0:
  print(f"Фильмы продолжительностью менее {min_duration_minutes} минут:")
  print(*short_movies, sep='\n')
else:
  print(f"Фильмы продолжительностью менее {min_duration_minutes} минут не найдены!")

Фильмы продолжительностью менее 120 минут:
Один дома
